In [ ]:
# import adatas and concatenate them. And see that the annotations are similar.


In [ ]:
# preprocess BCC 

import pandas as pd
import scanpy as sc

DATA_DIR = "/home/roger/protocol/cancer-pseudotime-grn-workflow/scripts/data/GSE123813"

# load metadata
metadata = pd.read_csv(
    f"{DATA_DIR}/GSE123813_bcc_all_metadata.txt.gz",
    sep="\t"
)

# load counts
counts = pd.read_csv(
    f"{DATA_DIR}/GSE123813_bcc_scRNA_counts.txt.gz",
    sep="\t",
    index_col=0
)

# transpose counts (cells x genes)
adata_bcc = sc.AnnData(counts.T)

# set cell IDs in metadata
metadata = metadata.set_index(metadata.columns[0])

# align metadata to adata cells
metadata = metadata.loc[adata_bcc.obs_names]

# assign metadata
adata_bcc.obs = metadata

In [ ]:
DATA_DIR = "/home/roger/protocol/cancer-pseudotime-grn-workflow/scripts/data/GSE123813"

metadata = pd.read_csv(f"{DATA_DIR}/GSE123813_scc_metadata.txt.gz", sep="\t")
counts = pd.read_csv(f"{DATA_DIR}/GSE123813_scc_scRNA_counts.txt.gz", sep="\t", index_col=0)

# create adata object
adata_scc = sc.AnnData(counts.T)
adata_scc.obs = metadata.set_index(metadata.columns[0]).loc[adata_scc.obs_names]

In [ ]:
# concatenate both adatas.

In [ ]:
adata_scc.obs["dataset"] = "SCC"
adata_bcc.obs["dataset"] = "BCC"

In [ ]:
import anndata as ad

adata = ad.concat(
    [adata_scc, adata_bcc],
    label="dataset",          # creates column
    keys=["SCC", "BCC"],      # labels
    join="inner",             # keep all genes  # if we do inner, we just keep the interesction
    merge="same"
)


In [ ]:
sc.pp.filter_cells(adata, min_genes=200) 
sc.pp.filter_genes(adata, min_cells=3)
adata

In [ ]:
del adata_bcc
del adata_scc

In [ ]:
#ad = adata.copy()

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True)
#ad = ad[:, ad.var["highly_variable"]].copy()
sc.pp.scale(adata)
sc.tl.pca(adata, n_comps=16)
sc.pp.neighbors(adata, n_pcs=16)
sc.tl.leiden(adata, resolution=1, flavor="igraph", n_iterations=2)
sc.tl.umap(adata) # we can change min_dist?!?!


In [ ]:
sc.pl.umap(adata, color=["leiden", "cluster", "dataset"])